# irr_pair11 Evaluation

Comprehensive evaluation of the irr_pair11 ansatz discovered via irreducible
ZX motif composition. Covers circuit structure, generalization to arbitrary
qubit counts, VQE benchmarking against baselines, statistical significance,
ablation study, and ZX-calculus analysis.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

from qiskit import QuantumCircuit
from qiskit.quantum_info import Operator

from zx_motifs.pipeline.ansatz import (
    irr_pair11_entangler, irr_pair11_original_6q,
    cx_chain_entangler, hea_entangler, build_hamiltonian,
)
from zx_motifs.pipeline.evaluation import vqe_test, count_2q, compute_entangling_power

%matplotlib inline

## 1. The irr_pair11 Circuit

The original irr_pair11 was discovered by composing two irreducible ZX motifs
(`phase_gadget_3t` + `cluster_chain`) into a 6-qubit entangling layer. Three
structural elements were identified as critical:

1. **Star hub** — central qubit entangled via CX fan-out to ~n/3 neighbours
2. **Phase gadgets** — T gates conjugated by CX pairs (irreducible non-Clifford)
3. **Chain tail** — nearest-neighbour CX chain extending entanglement to far qubits

In [ ]:
# Original 6q circuit
orig_6q = irr_pair11_original_6q()
print("Original 6q irr_pair11:")
print(f"  {orig_6q.num_qubits}q, {orig_6q.size()} gates, depth={orig_6q.depth()}")
print(f"  2Q gates: {count_2q(orig_6q)}")
print()
print(orig_6q.draw(output="text"))

## 2. Generalization

The generalized `irr_pair11_entangler(n)` scales all three structural elements
linearly with qubit count, ensuring every qubit participates.

In [ ]:
print("Circuit structure at each qubit count:")
print("-" * 60)
for nq in [4, 5, 6, 7, 8, 10]:
    qc = irr_pair11_entangler(nq)
    n_2q = count_2q(qc)
    n_1q = qc.size() - n_2q
    print(f"  {nq}q: {qc.size()} gates ({n_1q} 1Q + {n_2q} 2Q), depth={qc.depth()}")

print()
print("Generalized 6q circuit:")
gen_6q = irr_pair11_entangler(6)
print(gen_6q.draw(output="text"))

## 3. VQE Benchmark

Compare generalized irr_pair11 against CX-chain and HEA baselines across
multiple qubit counts and Hamiltonian models. Baselines use the same 2-qubit
gate budget as irr_pair11 for fair comparison.

In [ ]:
QUBIT_SIZES = [4, 5, 6]
MODELS = ["heisenberg", "tfim"]
N_RESTARTS = 10
MAXITER = 400

results = {}
for nq in QUBIT_SIZES:
    gen_qc = irr_pair11_entangler(nq)
    n_2q = count_2q(gen_qc)
    cx_qc = cx_chain_entangler(nq, max(1, n_2q))
    hea_qc = hea_entangler(nq, max(1, n_2q))

    for model in MODELS:
        H = build_hamiltonian(nq, model)
        evals = np.linalg.eigvalsh(H)
        exact_gs = float(evals[0])
        if exact_gs == 0:
            continue

        gen_res = vqe_test(gen_qc, nq, H, N_RESTARTS, MAXITER)
        cx_res = vqe_test(cx_qc, nq, H, N_RESTARTS, MAXITER)
        hea_res = vqe_test(hea_qc, nq, H, N_RESTARTS, MAXITER)

        gen_err = abs(gen_res["best_energy"] - exact_gs) / abs(exact_gs)
        cx_err = abs(cx_res["best_energy"] - exact_gs) / abs(exact_gs)
        hea_err = abs(hea_res["best_energy"] - exact_gs) / abs(exact_gs)

        best_base = min(cx_err, hea_err)
        wins = gen_err < best_base

        key = f"{nq}q_{model}"
        results[key] = {
            "n_qubits": nq, "model": model, "n_2q": n_2q,
            "gen_err": gen_err, "cx_err": cx_err, "hea_err": hea_err,
            "wins": wins, "exact_gs": exact_gs,
            "gen_energy": gen_res["best_energy"],
        }

        tag = "WIN" if wins else "lose"
        print(f"  {key}: gen={gen_err:.4f}  cx={cx_err:.4f}  hea={hea_err:.4f}  [{tag}]")

In [ ]:
print(f"\n{'Config':<20s} {'Gen':>8s} {'CX':>8s} {'HEA':>8s} {'Win?':>5s}")
print("-" * 50)
for key in sorted(results):
    r = results[key]
    tag = "YES" if r["wins"] else ""
    print(f"{key:<20s} {r['gen_err']:>8.4f} {r['cx_err']:>8.4f} {r['hea_err']:>8.4f} {tag:>5s}")

total_wins = sum(1 for r in results.values() if r["wins"])
print(f"\nOverall: {total_wins}/{len(results)} wins")

## 4. Baseline Comparison Plot

In [ ]:
configs = sorted(results.keys())
gen_errs = [results[k]["gen_err"] for k in configs]
cx_errs = [results[k]["cx_err"] for k in configs]
hea_errs = [results[k]["hea_err"] for k in configs]

x = np.arange(len(configs))
width = 0.25

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(x - width, gen_errs, width, label="irr_pair11", color="#4daf4a")
ax.bar(x, cx_errs, width, label="CX-chain", color="#377eb8")
ax.bar(x + width, hea_errs, width, label="HEA", color="#e41a1c")
ax.set_xticks(x)
ax.set_xticklabels(configs, rotation=45, ha="right")
ax.set_ylabel("Relative Error")
ax.set_title("VQE Performance: irr_pair11 vs Baselines")
ax.legend()
ax.set_yscale("log")
plt.tight_layout()
plt.show()

## 5. Statistical Significance

For winning configurations, run multiple seeds and compute t-test p-values
and Cohen's d effect size.

In [ ]:
winning_keys = [k for k, r in results.items() if r["wins"]]

for key in winning_keys:
    r = results[key]
    nq = r["n_qubits"]
    model = r["model"]
    gen_qc = irr_pair11_entangler(nq)
    n_2q = count_2q(gen_qc)
    best_base_name = "cx_chain" if r["cx_err"] < r["hea_err"] else "hea"
    best_base_qc = (cx_chain_entangler(nq, max(1, n_2q)) if best_base_name == "cx_chain"
                    else hea_entangler(nq, max(1, n_2q)))

    H = build_hamiltonian(nq, model)
    exact_gs = r["exact_gs"]

    gen_errors, base_errors = [], []
    for seed in range(42, 52):
        g = vqe_test(gen_qc, nq, H, N_RESTARTS, MAXITER, seed=seed)
        b = vqe_test(best_base_qc, nq, H, N_RESTARTS, MAXITER, seed=seed)
        gen_errors.append(abs(g["best_energy"] - exact_gs) / abs(exact_gs))
        base_errors.append(abs(b["best_energy"] - exact_gs) / abs(exact_gs))

    gen_arr = np.array(gen_errors)
    base_arr = np.array(base_errors)
    t_stat, p_val = stats.ttest_ind(gen_arr, base_arr, alternative="less")
    cohens_d = (np.mean(base_arr) - np.mean(gen_arr)) / np.sqrt(
        (np.std(gen_arr)**2 + np.std(base_arr)**2) / 2)

    sig = "SIGNIFICANT" if p_val < 0.05 else "not significant"
    print(f"{key}: gen={np.mean(gen_arr):.4f}\u00b1{np.std(gen_arr):.4f} "
          f"vs {best_base_name}={np.mean(base_arr):.4f}\u00b1{np.std(base_arr):.4f} "
          f"p={p_val:.6f} d={cohens_d:.2f} [{sig}]")

if not winning_keys:
    print("No winning configurations to test.")

## 6. Ablation Study

Remove each structural element (star hub, phase gadgets, chain tail) and
measure VQE degradation to confirm all three are necessary.

In [ ]:
nq = 6
H = build_hamiltonian(nq, "heisenberg")
exact_gs = float(np.linalg.eigvalsh(H)[0])

def no_hub(n):
    """irr_pair11 without star hub."""
    qc = QuantumCircuit(n)
    q = 1
    gadget_anchors = []
    while q < n - 1:
        anchor, target = q, q + 1
        if gadget_anchors:
            qc.cx(gadget_anchors[-1], anchor)
        qc.cx(anchor, target)
        qc.t(target)
        qc.cx(anchor, target)
        gadget_anchors.append(anchor)
        q += 3
    last = max(gadget_anchors[-1] + 1, max(2, n // 3)) if gadget_anchors else max(2, n // 3)
    for i in range(last, n - 1):
        qc.cx(i, i + 1)
    return qc

def no_phase(n):
    """irr_pair11 without phase gadgets (T gates removed)."""
    qc = QuantumCircuit(n)
    hub_size = max(2, n // 3)
    for i in range(1, hub_size + 1):
        qc.cx(i, 0)
    q = 1
    gadget_anchors = []
    while q < n - 1:
        anchor, target = q, q + 1
        if gadget_anchors:
            qc.cx(gadget_anchors[-1], anchor)
        qc.cx(anchor, target)
        qc.cx(anchor, target)  # no T gate
        gadget_anchors.append(anchor)
        q += 3
    last = max(gadget_anchors[-1] + 1, hub_size) if gadget_anchors else hub_size
    for i in range(last, n - 1):
        qc.cx(i, i + 1)
    return qc

def no_chain(n):
    """irr_pair11 without chain tail."""
    qc = QuantumCircuit(n)
    hub_size = max(2, n // 3)
    for i in range(1, hub_size + 1):
        qc.cx(i, 0)
    q = 1
    gadget_anchors = []
    while q < n - 1:
        anchor, target = q, q + 1
        if gadget_anchors:
            qc.cx(gadget_anchors[-1], anchor)
        qc.cx(anchor, target)
        qc.t(target)
        qc.cx(anchor, target)
        gadget_anchors.append(anchor)
        q += 3
    return qc  # no chain tail

ablations = {
    "full": irr_pair11_entangler,
    "no_hub": no_hub,
    "no_phase_gadgets": no_phase,
    "no_chain_tail": no_chain,
}

print(f"Ablation study ({nq}q Heisenberg, exact_gs={exact_gs:.4f})")
print("-" * 60)
for name, fn in ablations.items():
    qc = fn(nq)
    res = vqe_test(qc, nq, H, n_restarts=10, maxiter=400)
    err = abs(res["best_energy"] - exact_gs) / abs(exact_gs)
    print(f"  {name:20s}: error={err:.4f}  energy={res['best_energy']:.4f}  "
          f"gates={qc.size()}")

## 7. Entangling Power & ZX Analysis

Measure entangling power (average bipartite entropy over random inputs)
and ZX compression ratio.

In [ ]:
import pyzx as zx
from zx_motifs.pipeline.converter import qiskit_to_zx
import copy

for nq in [4, 6, 8]:
    qc = irr_pair11_entangler(nq)
    
    # Entangling power
    ep = compute_entangling_power(qc, n_samples=200)
    
    # ZX compression
    zx_circ = qiskit_to_zx(qc)
    g = zx_circ.to_graph()
    v_raw = g.num_vertices()
    g_reduced = copy.deepcopy(g)
    zx.simplify.full_reduce(g_reduced)
    v_reduced = g_reduced.num_vertices()
    compression = 1 - v_reduced / v_raw if v_raw > 0 else 0
    
    print(f"{nq}q: EP={ep['entangling_power']:.3f}\u00b1{ep['epd']:.3f}  "
          f"ZX {v_raw}\u2192{v_reduced} vertices (compression={compression:.1%})")

## 8. Summary

| Metric | Result |
|--------|--------|
| Discovery strategy | Irreducible composition (Strategy 3) |
| Motif components | phase_gadget_3t + cluster_chain |
| Key structure | Star hub + phase gadgets + chain tail |
| Scaling | Linear with qubit count |
| Ablation | All three elements necessary |